# Bootstrap Colab — Gen 3 (LLMs via Ollama)

Notebook de orquestração para inferência zero-shot de LLMs em Colab
(T4 é suficiente para todos os modelos Q4_K_M — L4 só acelera).
Mantém o código reutilizável em `src/ptbr_market/gen3_llm.py` — este
notebook só faz setup (Ollama + modelos) e chama `scripts/run_gen3.py`.

**Pré-requisitos no Drive** (upload manual antes da primeira execução):

```
MyDrive/ptbr-market-classification/
  artifacts/
    splits/
      train.parquet
      val.parquet
      test.parquet
      metadata.json
```

Os runs vão escrever em `MyDrive/ptbr-market-classification/artifacts/runs/`
automaticamente (via `PTBR_ARTIFACTS_ROOT`). `predictions.csv` é
append-only com `flush+fsync` por linha — se o Colab cair, basta
reiniciar e executar a célula de resume.

**Runtime recomendado:** GPU T4 (grátis) para os modelos Q4_K_M dessa
matriz. `qwen2.5:14b` (≈9 GB) é o mais apertado — se `ollama` relatar
OOM, troque para `qwen2.5:7b`.

**Tempo esperado** (test completo ≈16k amostras, 1 token gerado):
- Tucano 2.4B / Bode 7B: ~1-2 h por variante
- Llama 3.1 8B / Qwen 7B / Gemma 9B: ~2-4 h por variante
- Qwen 14B: ~4-6 h por variante

A etapa completa (6 modelos × 2 variantes) passa das 12 h livres do
Colab Free — planeje rodar um modelo por sessão, usando `--resume-run-id`
para retomar se necessário.

## 1. Montar o Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar o repositório

Clone fresco (não persiste no Drive) — garante exatamente o commit da
branch escolhida.

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification.git'
BRANCH = 'main'

!rm -rf /content/ptbr-market-classification
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/ptbr-market-classification
%cd /content/ptbr-market-classification
!git log -1 --oneline

## 3. Instalar dependências Python

Projeto em modo editável + extras `gen3` (`requests`, `huggingface-hub`).
O Ollama em si é instalado como binário na próxima célula.

In [ ]:
!pip install -q -e .
!pip install -q 'requests>=2.31' 'huggingface-hub>=0.24'

In [ ]:
!nvidia-smi | head -20

## 4. Instalar e iniciar o Ollama

Instala o binário oficial e inicia o daemon em background. `OLLAMA_MODELS`
aponta para `/content/ollama_models/` (efêmero no Colab, mas OK — puxar
de novo na próxima sessão custa ~2 min por modelo). Use `OLLAMA_HOST=0.0.0.0:11434`
só se for consumir de fora do Colab.

In [ ]:
import os, subprocess, time

os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
os.makedirs('/content/ollama_models', exist_ok=True)

!curl -fsSL https://ollama.com/install.sh | sh

# Sobe o daemon em background e espera ele responder
_ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/content/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
for _ in range(30):
    try:
        r = subprocess.run(['curl', '-sf', 'http://127.0.0.1:11434/'], capture_output=True)
        if r.returncode == 0:
            break
    except Exception:
        pass
    time.sleep(1)
!curl -sf http://127.0.0.1:11434/ && echo 'Ollama OK'

## 5. Apontar artefatos para o Drive

`PTBR_ARTIFACTS_ROOT` faz com que `new_run_dir` grave em
`artifacts/runs/` **dentro do Drive** — os runs sobrevivem ao
encerramento da sessão. O `predictions.csv` é append-only, então
retomar não perde trabalho.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/ptbr-market-classification'
os.environ['PTBR_ARTIFACTS_ROOT'] = f'{DRIVE_ROOT}/artifacts'
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'

!ls -la $PTBR_ARTIFACTS_ROOT/splits/

## 6. Puxar modelos nativos (Llama / Qwen / Gemma)

Puxe só os modelos que você vai rodar nesta sessão — cada `ollama pull`
gasta ~2-10 min dependendo do tamanho. As tags abaixo batem com
`GEN3_MODELS` em `gen3_llm.py`.

In [ ]:
!ollama pull llama3.1:8b-instruct-q4_K_M
# !ollama pull qwen2.5:7b-instruct-q4_K_M
# !ollama pull qwen2.5:14b-instruct-q4_K_M
# !ollama pull gemma2:9b-instruct-q4_K_M

## 7. Importar modelos pt-BR (Bode / Tucano) via GGUF

Esses modelos não estão na registry oficial do Ollama — precisam ser
baixados do HuggingFace como `.gguf` e importados com `ollama create`
a partir de um Modelfile minimalista (`FROM ./arquivo.gguf`).

Fontes:
- **Bode 7B**: `recogna-nlp/bode-7b-alpaca-pt-br-gguf` (repositório oficial)
- **Tucano 2.4B**: `tensorblock/Tucano-2b4-Instruct-GGUF` (mirror GGUF de `TucanoBR/Tucano-2b4-Instruct`)

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

def import_gguf(repo_id: str, filename: str, ollama_tag: str) -> None:
    local = hf_hub_download(repo_id=repo_id, filename=filename, local_dir='/content/gguf')
    mf = Path('/content/gguf') / f'Modelfile.{ollama_tag.replace(":", "_")}'
    mf.write_text(f'FROM {local}\nPARAMETER temperature 0\nPARAMETER stop "\\n"\n')
    print(f'Creating ollama model {ollama_tag} from {filename}...')
    import subprocess
    subprocess.run(['ollama', 'create', ollama_tag, '-f', str(mf)], check=True)

In [ ]:
# Bode 7B — ~4 GB de download
import_gguf(
    repo_id='recogna-nlp/bode-7b-alpaca-pt-br-gguf',
    filename='bode-7b-alpaca-q4_k_m.gguf',
    ollama_tag='bode-7b-alpaca:q4_k_m',
)

In [ ]:
# Tucano 2.4B — ~1.5 GB de download
import_gguf(
    repo_id='tensorblock/Tucano-2b4-Instruct-GGUF',
    filename='Tucano-2b4-Instruct-Q4_K_M.gguf',
    ollama_tag='tucano-2b4-instruct:q4_k_m',
)

In [ ]:
!ollama list

## 8. Rodar um modelo (uma célula por modelo × variante)

Cada modelo roda **duas vezes**: binária e multiclasse (`top7_plus_other`).
A comparação bin×mc8 é o mesmo protocolo de Gen 1/Gen 2 e responde à
pergunta "o prompt multiclasse ajuda o LLM?".

**Smoke test primeiro** (recomendado na primeira execução): rode com
`--limit 100` para confirmar o fluxo antes de gastar horas no corpus
completo.

In [ ]:
# Smoke test (≈2 min): 100 amostras só para validar o pipeline.
!python scripts/run_gen3.py --model llama3.1-8b --limit 100

In [ ]:
# Llama 3.1 8B — bin + mc8
!python scripts/run_gen3.py --model llama3.1-8b
!python scripts/run_gen3.py --model llama3.1-8b --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# # Qwen 2.5 7B
# !python scripts/run_gen3.py --model qwen2.5-7b
# !python scripts/run_gen3.py --model qwen2.5-7b --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# # Qwen 2.5 14B (apertado em T4 — se der OOM, use L4 ou pule para 7B)
# !python scripts/run_gen3.py --model qwen2.5-14b
# !python scripts/run_gen3.py --model qwen2.5-14b --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# # Gemma 2 9B
# !python scripts/run_gen3.py --model gemma2-9b
# !python scripts/run_gen3.py --model gemma2-9b --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# # Bode 7B (requer célula 7)
# !python scripts/run_gen3.py --model bode-7b
# !python scripts/run_gen3.py --model bode-7b --target-mode multiclass --collapse-scheme top7_plus_other

In [ ]:
# # Tucano 2.4B (requer célula 7)
# !python scripts/run_gen3.py --model tucano-2b4
# !python scripts/run_gen3.py --model tucano-2b4 --target-mode multiclass --collapse-scheme top7_plus_other

## 9. Retomar um run interrompido

Se a sessão Colab caiu no meio: reexecute as células 1-5 e 6/7 para ter
Ollama + modelo disponíveis de novo, depois aponte `--resume-run-id`
para o diretório do run. O `predictions.csv` já gravado é respeitado
— só os índices faltantes vão pro LLM.

In [ ]:
# Liste runs incompletos (sem metadata.json ou com predictions.csv curto):
import json
from pathlib import Path
import pandas as pd

runs_dir = Path(os.environ['PTBR_ARTIFACTS_ROOT']) / 'runs'
for d in sorted(runs_dir.iterdir()):
    if '__gen3__' not in d.name:
        continue
    pred = d / 'predictions.csv'
    meta = d / 'metadata.json'
    n_done = len(pd.read_csv(pred)) if pred.exists() else 0
    status = 'ok' if meta.exists() else 'INCOMPLETO'
    print(f'{status:12s} {n_done:>6d} linhas  {d.name}')

In [ ]:
# Ex.: retomar um run específico
# RUN_ID = '20260422-120000__gen3__llama3.1-8b__zs-v1-bin'
# !python scripts/run_gen3.py --model llama3.1-8b --resume-run-id $RUN_ID

## 10. Conferir o último run

In [ ]:
latest = max(
    (d for d in runs_dir.iterdir() if d.is_dir() and '__gen3__' in d.name),
    key=lambda d: d.stat().st_mtime,
)
print(f'Latest Gen 3 run: {latest.name}')
meta = json.loads((latest / 'metadata.json').read_text())
print(json.dumps(meta['metrics'], indent=2))
print(json.dumps(meta['efficiency'], indent=2))
print('scoring counts:', meta['config']['scoring']['method_counts'])
print('label counts:  ', meta['config']['scoring']['label_counts'])

## 11. Sincronizar localmente (depois, no host)

Como os runs ficam no Drive, basta baixá-los via Google Drive Desktop
ou `gdown`/`rclone` no host:

```bash
rclone sync drive:ptbr-market-classification/artifacts/runs ./artifacts/runs \
    --include '*__gen3__*'
```

Depois de baixar, gere os relatórios Gen 3:

```bash
uv run python scripts/gen3_report.py
```